# 🔄 Espelhamento de Imagens de Patologias — Data Augmentation

**Disciplina:** IA Aplicada à Construção Civil  
**Módulo:** Visão Computacional — Preparação de Datasets

---

## Contexto

Treinar um modelo de detecção de patologias (trincas, corrosão, segregação) exige centenas ou milhares de imagens anotadas. Em obras reais, esse volume raramente está disponível — o que leva ao **overfitting**: o modelo memoriza as imagens de treino e falha em campo.

A técnica mais simples e eficaz para contornar isso é o **Data Augmentation**: gerar variações sintéticas das imagens existentes sem alterar o conteúdo semântico. O espelhamento (flip) é o ponto de partida porque:

- **Não distorce geometria** — uma trinca continua sendo uma trinca espelhada
- **Dobra ou triplica o dataset** com custo computacional mínimo
- **Ensina invariância de orientação** ao modelo — ele aprende que trincas aparecem em qualquer direção

### Flip horizontal vs. vertical

| Tipo | `cv2.flip` | Quando usar |
|------|-----------|-------------|
| Horizontal (esquerda↔direita) | `flipCode=1` | Sempre válido para fachadas, lajes, vigas |
| Vertical (cima↔baixo) | `flipCode=0` | Válido para lajes e pisos; cuidado com fachadas (inverte gravidade visual) |
| Ambos (`hv`) | aplica os dois | Máximo de augmentation; use se o modelo não depende de orientação absoluta |

> 💡 **Atenção ao dataset:** se suas anotações estiverem em formato YOLO (`.txt` com coordenadas normalizadas) ou COCO (JSON), as coordenadas dos bounding boxes e máscaras também precisam ser espelhadas. Este notebook trata apenas das **imagens** — a transformação das anotações é o próximo módulo.

---

## Estrutura deste notebook

| Seção | O que você vai fazer |
|-------|----------------------|
| 1 | Testar o ambiente com imagem sintética (sem Drive) |
| 2 | Montar o Google Drive e configurar caminhos |
| 3 | Processar todas as imagens em lote |
| 4 | Visualizar resultados e conferir o dataset gerado |

> 💡 **Recomendação:** execute a Seção 1 primeiro — ela não depende de nenhum arquivo externo.

---
## Seção 1 — Teste com imagem sintética

Criamos uma imagem que simula uma foto de inspeção de trinca:

- **Fundo cinza** → superfície de concreto
- **Linha diagonal escura** → trinca principal
- **Ramificação** → trinca secundária (padrão de mapa)
- **Texto `REF`** → referência visual para checar se o espelhamento funcionou

**Resultado esperado:**
- Flip H: texto `REF` aparece espelhado (`ꓤƎЯ`) e a trinca vai para o lado oposto
- Flip V: imagem de cabeça para baixo
- Flip HV: ambos aplicados

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ── Criar imagem sintética 400×600 (BGR)
h, w = 400, 600
img_sint = np.full((h, w, 3), 160, dtype=np.uint8)  # concreto cinza

# Ruído de textura
ruido = np.random.normal(0, 8, img_sint.shape).astype(np.int16)
img_sint = np.clip(img_sint.astype(np.int16) + ruido, 0, 255).astype(np.uint8)

# Trinca principal (diagonal)
for i in range(50, 350):
    j = int(80 + i * 0.8)
    if j < w:
        cv2.line(img_sint, (j, i), (j+2, i+1), (40, 40, 40), 2)

# Ramificação
cv2.line(img_sint, (240, 200), (290, 260), (50, 50, 50), 1)
cv2.line(img_sint, (290, 260), (340, 300), (50, 50, 50), 1)

# Texto de referência para checar espelhamento
cv2.putText(img_sint, 'REF', (20, 50),
            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (30, 30, 200), 2, cv2.LINE_AA)

# ── Gerar os três flips
flip_h  = cv2.flip(img_sint, 1)   # horizontal
flip_v  = cv2.flip(img_sint, 0)   # vertical
flip_hv = cv2.flip(img_sint, -1)  # ambos

# ── Visualizar
fig, eixos = plt.subplots(1, 4, figsize=(20, 5))
imagens = [img_sint, flip_h, flip_v, flip_hv]
titulos = [
    'Original',
    'Flip Horizontal\n(flipCode=1)',
    'Flip Vertical\n(flipCode=0)',
    'Flip HV\n(flipCode=-1)'
]

for ax, im, t in zip(eixos, imagens, titulos):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(t, fontsize=10)
    ax.axis('off')

plt.suptitle('✅ Teste de ambiente — imagem sintética de trinca', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('✅ Ambiente OK — o texto REF deve aparecer espelhado no Flip H.')
print('   Pode prosseguir para a Seção 2.')

---
## Seção 2 — Montar o Google Drive e configurar

> ⚠️ **Pasta compartilhada?** Se `fotos_obra` não estiver em *Meu Drive*:  
> *Drive → clique direito → Organizar → Adicionar atalho ao Drive*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive montado.')

In [ ]:
from pathlib import Path

# ── Caminhos
PASTA_ENTRADA = Path('/content/drive/MyDrive/fotos_obra')
PASTA_SAIDA   = Path('/content/drive/MyDrive/fotos_obra_flip')
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

# ── Modo de espelhamento
# 'h'  → apenas horizontal
# 'v'  → apenas vertical
# 'hv' → ambos (dobra/triplica o dataset)
MODO = 'hv'

# ── Manter estrutura de subpastas na saída?
KEEP_STRUCTURE = False

# ── Extensões aceitas
EXTENSOES = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

# ── Verificar pasta
if not PASTA_ENTRADA.exists():
    print(f'⛔ Pasta não encontrada: {PASTA_ENTRADA}')
else:
    arquivos = [p for p in PASTA_ENTRADA.rglob('*') if p.suffix.lower() in EXTENSOES]
    n_out = len(arquivos) * (2 if MODO == 'hv' else 1)
    print(f'📂 Entrada  : {PASTA_ENTRADA}')
    print(f'📁 Saída    : {PASTA_SAIDA}')
    print(f'⚙️  Modo     : {MODO.upper()}')
    print(f'🖼️  {len(arquivos)} imagem(ns) → {n_out} arquivo(s) gerado(s)')

---
## Seção 3 — Processamento em lote

### Como o `cv2.flip` funciona

```
cv2.flip(img, flipCode)
              │
              ├── flipCode =  1  → espelha no eixo Y (esquerda ↔ direita)
              ├── flipCode =  0  → espelha no eixo X (cima ↔ baixo)
              └── flipCode = -1  → ambos simultaneamente
```

O resultado é salvo com sufixo `_flipH`, `_flipV` ou ambos, preservando a extensão original.

### Nomenclatura dos arquivos gerados

```
fotos_obra/
  trinca_001.jpg
        │
        ├── fotos_obra_flip/trinca_001_flipH.jpg
        └── fotos_obra_flip/trinca_001_flipV.jpg
```

Se `KEEP_STRUCTURE = True`, subpastas são replicadas na saída — útil quando o dataset já está organizado por classe (`trincas/`, `corrosao/`, etc.).

In [ ]:
from tqdm.notebook import tqdm

gerados = []
erros   = []

for p in tqdm(arquivos, desc='Espelhando imagens'):
    try:
        img = cv2.imread(str(p), cv2.IMREAD_COLOR)
        if img is None:
            erros.append((p.name, 'Falha ao ler'))
            continue

        # Destino
        rel     = p.relative_to(PASTA_ENTRADA)
        dst_dir = PASTA_SAIDA / (rel.parent if KEEP_STRUCTURE else Path())
        dst_dir.mkdir(parents=True, exist_ok=True)

        stem = p.stem
        ext  = p.suffix.lower()

        if MODO in ('h', 'hv'):
            saida = dst_dir / f'{stem}_flipH{ext}'
            cv2.imwrite(str(saida), cv2.flip(img, 1))
            gerados.append(saida.name)

        if MODO in ('v', 'hv'):
            saida = dst_dir / f'{stem}_flipV{ext}'
            cv2.imwrite(str(saida), cv2.flip(img, 0))
            gerados.append(saida.name)

    except Exception as e:
        erros.append((p.name, str(e)))

print(f'\n✅ {len(gerados)} arquivo(s) gerado(s) em: {PASTA_SAIDA}')
print(f'   Dataset original : {len(arquivos)} imagens')
print(f'   Dataset aumentado: {len(arquivos) + len(gerados)} imagens')

if erros:
    print('\n⚠️  Arquivos com erro:')
    for nome, msg in erros:
        print(f'   {nome}: {msg}')

---
## Seção 4 — Visualização e conferência do dataset

### 4a. Painel Original × Flip H × Flip V

Confira visualmente se o espelhamento preservou as características da patologia. Para trincas, observe:
- A **morfologia da trinca** (comprimento, largura, ramificações) deve ser idêntica
- Apenas a **orientação** muda
- Não deve haver **artefatos de borda** ou perda de qualidade

In [ ]:
MAX_EXIBIR = 4
amostra = arquivos[:MAX_EXIBIR]

for p in amostra:
    img = cv2.imread(str(p), cv2.IMREAD_COLOR)
    if img is None:
        continue

    variantes = [('Original', img)]
    if MODO in ('h', 'hv'):
        variantes.append(('Flip H', cv2.flip(img, 1)))
    if MODO in ('v', 'hv'):
        variantes.append(('Flip V', cv2.flip(img, 0)))

    fig, eixos = plt.subplots(1, len(variantes), figsize=(6 * len(variantes), 5))
    if len(variantes) == 1:
        eixos = [eixos]

    for ax, (titulo, im) in zip(eixos, variantes):
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(titulo, fontsize=10)
        ax.axis('off')

    plt.suptitle(p.name, fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 4b. Inventário do dataset gerado

Resumo do que foi produzido e fator de multiplicação do dataset.

In [ ]:
import pandas as pd
from pathlib import Path

# Listar tudo na pasta de saída
saidas = sorted([p for p in PASTA_SAIDA.rglob('*') if p.suffix.lower() in EXTENSOES])

df = pd.DataFrame({
    'Arquivo gerado': [p.name for p in saidas],
    'Tipo': ['Flip H' if '_flipH' in p.name else 'Flip V' for p in saidas],
    'Tamanho (KB)': [round(p.stat().st_size / 1024, 1) for p in saidas]
})

display(df)

fator = (len(arquivos) + len(gerados)) / max(len(arquivos), 1)
print(f'\n📊 Fator de aumento  : ×{fator:.1f}')
print(f'   Imagens originais : {len(arquivos)}')
print(f'   Imagens geradas   : {len(gerados)}')
print(f'   Total do dataset  : {len(arquivos) + len(gerados)}')

---
## Próximos passos

O espelhamento é apenas a primeira técnica de augmentation. O pipeline completo para um dataset robusto combina:

```
Imagens originais
        │
        ├── Flip H / V          ← este notebook  (×2 ou ×3)
        ├── Rotação ±15°        ← próximo módulo (×N)
        ├── Zoom / crop         ← próximo módulo
        ├── Ajuste de brilho    ← simula diferentes horas do dia
        └── Adição de ruído     ← simula câmeras de obra
```

Bibliotecas recomendadas para augmentation avançado:
- **Albumentations** — rápida, suporta transformação simultânea de imagem + máscara + bbox
- **imgaug** — flexível, boa documentação
- **torchvision.transforms** — integrado ao PyTorch, ideal se já está no pipeline de treino

### Referências

- Shorten, C.; Khoshgoftaar, T. M. (2019). *A survey on Image Data Augmentation for Deep Learning*. Journal of Big Data, 6(1), 60.
- OpenCV docs: [`cv2.flip`](https://docs.opencv.org/4.x/d2/de8/group__core__array.html#gaca7be533e3dac7feb70fc60635adf441)
- Buslaev, A. et al. (2020). *Albumentations: Fast and Flexible Image Augmentations*. Information, 11(2), 125.
